# 06 — Memory & Conversation

In notebook 02 we saw the core truth: **chat models are stateless**. To hold a
conversation, you resend the history every turn. Doing that by hand — appending each
`AIMessage`, threading the list through — gets tedious and bug-prone, especially when your
app serves many users at once.

This notebook shows how to automate it:

1. The manual baseline (a quick recap of the problem).
2. **`RunnableWithMessageHistory`** — wrap a chain so history is tracked automatically,
   per conversation.
3. Isolating **multiple sessions** (different users / threads).
4. **Trimming** history so long chats don't blow past the context window or your budget.
5. A note on **persistence and production**.

In [ ]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

load_dotenv()
assert os.environ.get("GOOGLE_API_KEY"), "Set GOOGLE_API_KEY in your .env file first."

model = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.3)
print("Model ready.")

## 1. The problem, restated

Here's the manual approach from notebook 02. It works, but *you* own the list: you must
append every reply, pass it around, and keep a separate list per user. Imagine doing this
inside a web server for thousands of conversations.

In [ ]:
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

history = [SystemMessage(content="You are a friendly assistant.")]
history.append(HumanMessage(content="My favorite language is Python."))
reply = model.invoke(history)
history.append(reply)   # don't forget this, or the model loses the turn

history.append(HumanMessage(content="What did I say my favorite language was?"))
print(model.invoke(history).content)

## 2. `RunnableWithMessageHistory` — automatic history

LangChain can manage that list for you. The recipe:

1. Build a chain whose prompt has a **`MessagesPlaceholder`** for the running history.
2. Provide a function that returns a *history store* for a given **session id**.
3. Wrap the chain in **`RunnableWithMessageHistory`**.

After that, you just call `.invoke({"input": ...}, config={...session_id...})` and the
wrapper reads past messages from the store, runs the chain, and writes the new messages
back — automatically.

In [ ]:
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

# 1. A prompt with a slot for history.
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a friendly assistant. Keep replies short."),
    MessagesPlaceholder("history"),
    ("human", "{input}"),
])
chain = prompt | model

# 2. A store: session_id -> message history. (In-memory here; swap for a DB in production.)
store = {}

def get_session_history(session_id: str) -> InMemoryChatMessageHistory:
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

# 3. Wrap the chain.
chatbot = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",     # which input key holds the new user message
    history_messages_key="history", # which placeholder receives past messages
)
print("Chatbot ready.")

Now talk to it. The only new thing is the `config` — it carries the `session_id` that
selects which conversation's history to use. Notice we never touch the message list.

In [ ]:
cfg = {"configurable": {"session_id": "user-1"}}

print(chatbot.invoke({"input": "Hi! My name is Alex and I love hiking."}, config=cfg).content)
print(chatbot.invoke({"input": "What's my name and hobby?"}, config=cfg).content)

The second answer recalls "Alex" and "hiking" even though we only sent the new
question — the wrapper supplied the history behind the scenes. Inspect the store to see
what it saved:

In [ ]:
for msg in store["user-1"].messages:
    print(f"[{msg.type}] {msg.content}")

## 3. Multiple sessions stay isolated

The `session_id` keys keep conversations separate, which is exactly what a multi-user app
needs. Start a second session and confirm it knows nothing about the first.

In [ ]:
cfg2 = {"configurable": {"session_id": "user-2"}}

print("user-2:", chatbot.invoke({"input": "Do you know my name?"}, config=cfg2).content)
print("user-1:", chatbot.invoke({"input": "Do you still know my name?"}, config=cfg).content)

`user-2` has no memory of Alex; `user-1` still does. Each session id maps to its own
history object in the store.

## 4. Trimming: don't let history grow forever

Every turn you resend the whole history, so a long conversation means more tokens (more
cost, slower responses) and can eventually exceed the model's context window. The fix is
to **trim** older messages before sending.

`trim_messages` does this with a few strategies. Below we keep only the most recent
messages. We count "tokens" as the number of messages (`token_counter=len`) to keep the
example dependency-free; in production you'd pass a real token counter so the limit
reflects actual context size.

In [ ]:
from langchain_core.messages import trim_messages

# A long pretend conversation.
long_history = [SystemMessage(content="You are a helpful assistant.")]
for i in range(1, 7):
    long_history.append(HumanMessage(content=f"Question {i}"))
    long_history.append(AIMessage(content=f"Answer {i}"))

trimmer = trim_messages(
    max_tokens=5,            # with token_counter=len, this means "about 5 messages"
    strategy="last",         # keep the most recent
    token_counter=len,
    include_system=True,     # always keep the system message
    start_on="human",        # the kept window should start on a human turn
)

trimmed = trimmer.invoke(long_history)
print(f"Trimmed {len(long_history)} messages down to {len(trimmed)}:")
for m in trimmed:
    print(f"  [{m.type}] {m.content}")

You wire the trimmer into the chain so it runs on every turn — placing it before the
prompt so the model only ever sees a bounded window plus the system message.

In [ ]:
# The trimmer is a Runnable, so it composes like everything else.
# It receives the message list assembled from history + the new input.
bounded_chain = (
    {"history": lambda x: trimmer.invoke(x["history"]), "input": lambda x: x["input"]}
    | prompt
    | model
)

# (For a full chatbot you'd wrap `bounded_chain` in RunnableWithMessageHistory exactly as
#  in section 2 — the only change is the trimmer in front.)
print("Bounded chain assembled.")

> **Other strategies.** `trim_messages` can also keep the *first* messages
> (`strategy="first"`), and there are summarization approaches where an LLM compresses old
> turns into a short recap instead of dropping them. Trimming is the simplest and most
> common starting point.

## 5. Persistence and production

`InMemoryChatMessageHistory` lives in a Python dict — it vanishes when the process
restarts. For real apps you back the store with a database so history survives:

- Swap `get_session_history` to read/write Redis, Postgres, SQLite, etc. The chain code
  doesn't change — only the store implementation does.
- For **agents** (notebook 10) and more complex stateful workflows, LangChain 1.0 builds
  on **LangGraph**, which has its own persistence layer (*checkpointers*). A checkpointer
  saves the whole graph state per thread and is the recommended path for production
  agents. We'll use it when we build agents.

The mental model stays constant throughout: **memory is just stored messages, reattached
to each request.**

## Your turn

Five exercises around conversations you'd actually run — a helpdesk bot keyed by ticket, a
multi-tenant isolation check, a tutor that tracks progress, a cost-bounded long chat, and
saving/restoring a conversation. Try each in a blank cell before opening its solution.

**Exercise 1 — Helpdesk bot keyed by ticket.** Using the `chatbot` from section 2, open a
session named after a support ticket. Have the customer describe a device and a symptom over
separate turns, then ask the bot to summarize the issue for the next agent.

*Sample run (illustrative — only the final summary is printed):*

```text
Summary for the next agent: The customer's Pixel 7 won't connect to Wi-Fi. They have
already tried restarting their router, but the issue persists.
```

<details><summary>Show solution</summary>

```python
cfg = {"configurable": {"session_id": "ticket-558"}}
chatbot.invoke({"input": "My Pixel 7 won't connect to Wi-Fi."}, config=cfg)
chatbot.invoke({"input": "I already tried restarting the router."}, config=cfg)
print(chatbot.invoke(
    {"input": "Summarize my issue and what I've tried, for the next agent."},
    config=cfg,
).content)
```

Using the ticket id as the `session_id` is exactly how you'd thread a helpdesk conversation.

</details>

**Exercise 2 — Multi-tenant isolation check.** Two customers talk to the same bot under
different session ids. Confirm what one says never leaks into the other's session.

*Sample run (illustrative):*

```text
B sees: I'm sorry, I don't have an account code on file for you. Could you share it?
A sees: Your account code is NORTH-99.
```

<details><summary>Show solution</summary>

```python
a = {"configurable": {"session_id": "tenant-A"}}
b = {"configurable": {"session_id": "tenant-B"}}

chatbot.invoke({"input": "Remember my account code is NORTH-99."}, config=a)
print("B sees:", chatbot.invoke({"input": "What's my account code?"}, config=b).content)
print("A sees:", chatbot.invoke({"input": "What's my account code?"}, config=a).content)
```

Tenant B knows nothing about A's code — the store keys conversations apart by session id.

</details>

**Exercise 3 — Tutor that tracks progress.** Open a session for a student, tell the bot which
lessons they've finished over a couple of turns, then ask it to recommend the next topic based
on that history.

*Sample run (illustrative):*

```text
Since you've covered variables, loops, and lists, a natural next step is dictionaries —
they'll let you store data as key-value pairs.
```

<details><summary>Show solution</summary>

```python
cfg = {"configurable": {"session_id": "student-42"}}
chatbot.invoke({"input": "I just finished the lesson on Python lists."}, config=cfg)
chatbot.invoke({"input": "Earlier I covered variables and loops."}, config=cfg)
print(chatbot.invoke(
    {"input": "Based on what I've covered, what should I learn next? One suggestion."},
    config=cfg,
).content)
```

The bot personalizes the recommendation because the earlier turns are in its memory.

</details>

**Exercise 4 — Cost-bounded long chat.** Wrap a *trimmer-bounded* chain in
`RunnableWithMessageHistory` so that, however long the chat grows, each call only sends a small
recent window to the model.

*Sample run (illustrative):*

```text
messages stored: 8
```

<details><summary>Show solution</summary>

```python
from langchain_core.runnables.history import RunnableWithMessageHistory

keep_recent = trim_messages(max_tokens=4, strategy="last", token_counter=len,
                            include_system=True, start_on="human")
bounded = (
    {"history": lambda x: keep_recent.invoke(x["history"]), "input": lambda x: x["input"]}
    | prompt | model
)

store2 = {}
def hist2(sid):
    return store2.setdefault(sid, InMemoryChatMessageHistory())

bot = RunnableWithMessageHistory(bounded, hist2,
        input_messages_key="input", history_messages_key="history")

cfg = {"configurable": {"session_id": "trip"}}
for q in ["I'm planning a trip to Japan.", "Mainly Tokyo and Kyoto.",
          "Two weeks in spring.", "Roughly what should I budget?"]:
    bot.invoke({"input": q}, config=cfg)

print("messages stored:", len(store2["trip"].messages))   # full history is kept on disk...
# ...but the trimmer means the model only ever sees the last few each turn.
```

The store remembers everything; the trimmer bounds what you actually pay to send.

</details>

**Exercise 5 — Save & restore a conversation.** Serialize a session's messages to plain dicts
(as you'd store in a database), then rebuild a fresh history object from them — the pattern
behind persistent chat.

*Sample run (illustrative):*

```text
[{'role': 'human', 'text': 'Hi! My name is Alex and I love hiking.'}, {'role': 'ai', 'text': "Nice to meet you, Alex! Hiking is a wonderful hobby."}, {'role': 'human', 'text': "What's my name and hobby?"}, {'role': 'ai', 'text': 'Your name is Alex and you love hiking.'}]
restored 4 messages
```

<details><summary>Show solution</summary>

```python
# Save: turn stored messages into DB-ready rows.
def dump(history):
    return [{"role": m.type, "text": m.content} for m in history.messages]

saved = dump(store["user-1"])   # from section 2
print(saved)

# Restore: rebuild a history object, as if loading it on the next request.
restored = InMemoryChatMessageHistory()
for row in saved:
    if row["role"] == "human":
        restored.add_user_message(row["text"])
    else:
        restored.add_ai_message(row["text"])
print("restored", len(restored.messages), "messages")
```

Swapping the in-memory store for Redis/Postgres is just changing where `dump`/restore read and
write — the chain stays the same.

</details>

## Summary

- Memory = **stored messages reattached to each request**; models stay stateless.
- **`RunnableWithMessageHistory`** automates history: give it a chain (with a
  `MessagesPlaceholder`) and a `get_session_history(session_id)` function.
- The **`session_id`** in `config` keeps multiple conversations isolated.
- **`trim_messages`** bounds history so long chats stay within the context window and budget.
- In-memory stores are for demos; back them with a database for persistence, and use
  **LangGraph checkpointers** for stateful agents.

**Next:** [07 — Document Loaders & Text Splitters](07_Document_Loaders_and_Text_Splitters.ipynb).
We start building toward retrieval: getting your own data into LangChain and chunking it well.